In [ ]:
# ⚠️ CRITICAL: Must run this FIRST and ONLY ONCE!
# This cell completely removes torchvision to prevent circular import errors
import subprocess
import sys
import os

print("⚠️  Step 1: Uninstalling torchvision completely...")
result = subprocess.run(
    ["pip", "uninstall", "-y", "torchvision"],
    capture_output=True,
    text=True,
    timeout=60
)
print(f"   {result.stdout.split(chr(10))[0]}")

print("\n✅ Step 2: Setting environment variables...")
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("   ✅ All environment variables set")

print("\n✅ Step 3: Installing import hook...")
# Remove any cached torchvision modules
modules_to_remove = [name for name in list(sys.modules.keys()) if 'torchvision' in name.lower()]
for module_name in modules_to_remove:
    del sys.modules[module_name]
print(f"   ✅ Removed {len(modules_to_remove)} cached torchvision modules")

# Block future imports
class BlockTorchvision:
    def find_module(self, fullname, path=None):
        if 'torchvision' in fullname.lower():
            raise ImportError("torchvision is permanently disabled")
        return None

sys.meta_path.insert(0, BlockTorchvision())
print("   ✅ Import hook installed")

print("\n" + "="*70)
print("✅ ALL TORCHVISION BLOCKS ACTIVATED")
print("="*70)
print("\n⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel")
print("   and run this cell again as the VERY FIRST cell.\n")


⚠️  Step 1: Uninstalling torchvision completely...
   Found existing installation: torchvision 0.25.0+cu128

✅ Step 2: Setting environment variables...
   ✅ All environment variables set

✅ Step 3: Installing import hook...
   ✅ Removed 0 cached torchvision modules
   ✅ Import hook installed

✅ ALL TORCHVISION BLOCKS ACTIVATED

⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel
   and run this cell again as the VERY FIRST cell.



In [ ]:
# 安装依赖（约3-5分钟）
print("📦 Installing dependencies...")

# 先锁定关键基础包（避免自动升级）
!pip install -q torch==2.9.0 --no-deps
!pip install -q fsspec==2024.3.1
!pip install -q numpy==2.0.2 --no-deps

# 安装主要训练库（指定兼容版本）
!pip install -q transformers==4.46.0
!pip install -q peft==0.13.0
!pip install -q datasets==2.19.0
!pip install -q "accelerate>=1.0.0"
!pip install -q sentencepiece==0.2.0
!pip install -q tqdm
!pip install -q huggingface-hub==0.26.0

print("✅ Core dependencies installed! If running in Colab, restart runtime after this cell.")

📦 Installing dependencies...
✅ Core dependencies installed! If running in Colab, restart runtime after this cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, TaskType, get_peft_model

# ============================================================
# 模型与训练配置
# ============================================================

MODEL_NAME   = "codellama/CodeLlama-7b-Instruct-hf"
OUTPUT_DIR   = "/content/drive/MyDrive/gis-models/step-level-model-865"

# LoRA 参数（865条小数据集，适当提高 r 以增强拟合）
LORA_R       = 64
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# 训练参数
# 865条数据很精炼 → 用更多 epoch，batch 不需要很大
BATCH_SIZE              = 2
GRADIENT_ACCUMULATION   = 4      # 有效 batch = 8
NUM_EPOCHS              = 10     # 小数据集多训练几轮
LEARNING_RATE           = 2e-4
WEIGHT_DECAY            = 0.01
MAX_LENGTH              = 1024    # 步骤级数据 output 较短，但为了避免截断，增大长度

print("="*70)
print("🔧 模型与训练配置（865条去重数据专用）")
print("="*70)
print(f"  MODEL_NAME              : {MODEL_NAME}")
print(f"  OUTPUT_DIR              : {OUTPUT_DIR}")
print()
print(f"  LoRA  r / alpha         : {LORA_R} / {LORA_ALPHA}")
print(f"  LoRA  dropout           : {LORA_DROPOUT}")
print()
print(f"  BATCH_SIZE              : {BATCH_SIZE}")
print(f"  GRADIENT_ACCUMULATION   : {GRADIENT_ACCUMULATION}")
print(f"  有效 batch 大小          : {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  NUM_EPOCHS              : {NUM_EPOCHS}")
print(f"  MAX_LENGTH              : {MAX_LENGTH} tokens")
print(f"  LEARNING_RATE           : {LEARNING_RATE}")

🔧 模型与训练配置（865条去重数据专用）
  MODEL_NAME              : codellama/CodeLlama-7b-Instruct-hf
  OUTPUT_DIR              : /content/drive/MyDrive/gis-models/step-level-model-865

  LoRA  r / alpha         : 64 / 32
  LoRA  dropout           : 0.05

  BATCH_SIZE              : 2
  GRADIENT_ACCUMULATION   : 4
  有效 batch 大小          : 8
  NUM_EPOCHS              : 10
  MAX_LENGTH              : 1024 tokens
  LEARNING_RATE           : 0.0002


### 🚀 加载已微调模型进行推理

In [ ]:
# 设置模型路径
# OUTPUT_DIR (在之前的配置单元格中定义) 即为已训练模型的保存路径
model_path = OUTPUT_DIR
print(f"将从以下路径加载已微调模型: {model_path}")


将从以下路径加载已微调模型: /content/drive/MyDrive/gis-models/step-level-model-865


In [ ]:
import os

print(f"========== 读取目录内容: {model_path} ==========")
if os.path.exists(model_path):
    for root, dirs, files in os.walk(model_path):
        level = root.replace(model_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
else:
    print(f"目录不存在，请检查路径: {model_path}")
print("==========================================================")


========== 读取目录内容: /content/drive/MyDrive/gis-models/step-level-model-865 ==========
step-level-model-865/
    model.safetensors
    training_args.bin
    tokenizer_config.json
    special_tokens_map.json
    tokenizer.model
    added_tokens.json
    tokenizer.json
    training_info.json
    README.md
    tokenizer/
        tokenizer_config.json
        tokenizer.model
        special_tokens_map.json
        added_tokens.json
        tokenizer.json


In [ ]:
import json
import os

print("="*70)
print("🔍 深度诊断：检查所有配置文件")
print("="*70)

# 检查所有 .json 文件
print("\n📋 所有 JSON 文件内容:\n")

json_files = [
    "tokenizer_config.json",
    "training_info.json",
    "special_tokens_map.json",
    "added_tokens.json"
]

for json_file in json_files:
    file_path = os.path.join(model_path, json_file)
    if os.path.exists(file_path):
        print(f"\n{'─'*70}")
        print(f"📄 {json_file}:")
        print('─'*70)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                content = json.dumps(data, indent=2, ensure_ascii=False)
                # 如果太长就截断显示
                if len(content) > 1500:
                    print(content[:1500])
                    print(f"\n   ... (已截断，共 {len(content)} 字符)")
                else:
                    print(content)
        except Exception as e:
            print(f"❌ 读取失败: {e}")
    else:
        print(f"\n❌ 文件不存在: {json_file}")

# 检查 README.md
print(f"\n{'─'*70}")
print("📄 README.md 完整内容:")
print('─'*70)
readme_path = os.path.join(model_path, "README.md")
if os.path.exists(readme_path):
    try:
        with open(readme_path, "r", encoding="utf-8") as f:
            readme_content = f.read()
            if len(readme_content) > 2000:
                print(readme_content[:2000])
                print(f"\n... (已截断，共 {len(readme_content)} 字符)")
            else:
                print(readme_content)
    except Exception as e:
        print(f"❌ 读取失败: {e}")
else:
    print("❌ README.md 不存在")

# 检查 training_args.bin (二进制文件，尝试查看文件大小)
print(f"\n{'─'*70}")
print("📊 二进制文件信息:")
print('─'*70)
training_args_path = os.path.join(model_path, "training_args.bin")
if os.path.exists(training_args_path):
    size_mb = os.path.getsize(training_args_path) / (1024 * 1024)
    print(f"✅ training_args.bin 存在 (大小: {size_mb:.2f} MB)")
    print("   ℹ️  这是二进制文件，包含 TrainingArguments 对象")
else:
    print("❌ training_args.bin 不存在")

# 尝试用 torch 加载 training_args.bin
print("\n📥 尝试加载 training_args.bin 中的配置...")
try:
    import torch
    training_args = torch.load(training_args_path)
    print("✅ 成功加载! 类型:", type(training_args))

    # 如果是 TrainingArguments 对象，尝试找模型相关信息
    if hasattr(training_args, 'to_dict'):
        args_dict = training_args.to_dict()
        print("\n📋 training_args 的主要字段:")
        for key, value in sorted(args_dict.items()):
            if isinstance(value, str) and len(value) < 100:
                print(f"   • {key}: {value}")
            elif not isinstance(value, (dict, list)):
                print(f"   • {key}: {value}")
except Exception as e:
    print(f"⚠️  加载失败: {e}")

print("\n" + "="*70)
print("💡 诊断完成！根据上面的信息可以确定模型的基础模型名称")
print("="*70)


🔍 深度诊断：检查所有配置文件

📋 所有 JSON 文件内容:


──────────────────────────────────────────────────────────────────────
📄 tokenizer_config.json:
──────────────────────────────────────────────────────────────────────
{
  "add_bos_token": true,
  "add_eos_token": false,
  "added_tokens_decoder": {
    "0": {
      "content": "<unk>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "1": {
      "content": "<s>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "2": {
      "content": "</s>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "32007": {
      "content": "▁<PRE>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "32008": {
      "content": "▁<SUF>",
     

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from peft import PeftModel, LoraConfig, get_peft_model
import torch
import os
import json

print(f"开始加载模型和 Tokenizer (基础模型: {MODEL_NAME})...")

# 1. 加载 Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print("✅ 成功从本地路径加载 Tokenizer。")
except Exception as e:
    print(f"⚠️ 从本地加载 Tokenizer 失败。正在从基础模型加载...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("✅ 成功从基础模型加载 Tokenizer。")

# 2. 检查文件结构
adapter_config_path = os.path.join(model_path, "adapter_config.json")
model_safetensors = os.path.join(model_path, "model.safetensors")
adapter_bin = os.path.join(model_path, "adapter_model.bin")

print("\n📋 文件检查:")
print(f"   adapter_config.json: {'✅ 存在' if os.path.exists(adapter_config_path) else '❌ 不存在'}")
print(f"   model.safetensors: {'✅ 存在 ({:.1f}GB)'.format(os.path.getsize(model_safetensors)/(1024**3)) if os.path.exists(model_safetensors) else '❌ 不存在'}")
print(f"   adapter_model.bin: {'✅ 存在' if os.path.exists(adapter_bin) else '❌ 不存在'}")

# 3. 判断模型类型并加载
print("\n🔍 判断模型类型...")

# 先尝试直接加载为完整模型（如果 model.safetensors 是完全合并的模型）
if os.path.exists(model_safetensors) and not os.path.exists(adapter_config_path):
    print("💡 尝试方案1: 直接加载为完整合并模型...")
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print("✅ 方案1成功！模型是完全合并的完整模型。")
        MODEL_LOAD_SUCCESS = True
    except Exception as e:
        print(f"⚠️ 方案1失败: {str(e)[:200]}")
        MODEL_LOAD_SUCCESS = False
else:
    MODEL_LOAD_SUCCESS = False

# 如果方案1失败，尝试 LoRA 加载
if not MODEL_LOAD_SUCCESS:
    print("\n💡 尝试方案2: LoRA 加载...")
    try:
        # 加载基础模型
        print("   正在加载基础模型...")
        from transformers import BitsAndBytesConfig
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )

        if os.path.exists(adapter_config_path):
            # 使用标准 PEFT 加载
            print("   检测到 adapter_config.json，使用 PEFT...")
            model = PeftModel.from_pretrained(
                base_model,
                model_path,
                device_map="auto",
                torch_dtype=torch.float16
            )
        else:
            # 手动构建 LoRA
            print("   手动构建 LoRA 适配器...")
            training_info_path = os.path.join(model_path, "training_info.json")
            if os.path.exists(training_info_path):
                with open(training_info_path, "r", encoding="utf-8") as f:
                    training_info = json.load(f)
                lora_r = training_info.get("lora_r", 64)
                lora_alpha = training_info.get("lora_alpha", 32)
                lora_dropout = training_info.get("lora_dropout", 0.05)
            else:
                lora_r, lora_alpha, lora_dropout = 64, 32, 0.05

            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "v_proj"],
                lora_dropout=lora_dropout,
                bias="none",
                task_type="CAUSAL_LM"
            )
            model = get_peft_model(base_model, lora_config)

            # 尝试从 model.safetensors 加载权重
            if os.path.exists(model_safetensors):
                try:
                    from safetensors.torch import load_file
                    state_dict = load_file(model_safetensors)
                    # 只加载以 "lora" 开头的权重
                    lora_weights = {k: v for k, v in state_dict.items() if "lora" in k.lower()}
                    if lora_weights:
                        missing, unexpected = model.load_state_dict(lora_weights, strict=False)
                        print(f"   ✅ 加载了 {len(lora_weights)} 个 LoRA 权重")
                    else:
                        print(f"   ⚠️ 未找到 lora 相关权重 (检查了 {len(state_dict)} 个权重)")
                except Exception as e:
                    print(f"   ⚠️ 权重加载失败: {str(e)[:100]}")

        print("✅ 方案2成功！LoRA 模型已加载。")
        MODEL_LOAD_SUCCESS = True
    except Exception as e:
        print(f"❌ 方案2也失败: {str(e)[:300]}")
        MODEL_LOAD_SUCCESS = False

if MODEL_LOAD_SUCCESS:
    print("\n🎉 模型加载完成！")
else:
    print("\n❌ 模型加载失败，请检查错误信息。")


开始加载模型和 Tokenizer (基础模型: codellama/CodeLlama-7b-Instruct-hf)...
✅ 成功从本地路径加载 Tokenizer。

📋 文件检查:
   adapter_config.json: ❌ 不存在
   model.safetensors: ✅ 存在 (12.6GB)
   adapter_model.bin: ❌ 不存在

🔍 判断模型类型...
💡 尝试方案1: 直接加载为完整合并模型...
⚠️ 方案1失败: Unrecognized model in /content/drive/MyDrive/gis-models/step-level-model-865. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, alt

💡 尝试方案2: LoRA 加载...
   正在加载基础模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

   手动构建 LoRA 适配器...
   ✅ 加载了 128 个 LoRA 权重
✅ 方案2成功！LoRA 模型已加载。

🎉 模型加载完成！


In [ ]:
# 检查并设置 tokenizer 的 pad_token，以防其缺失，这对于生成很重要。
# 如果 tokenizer.pad_token_id 为 None，则将其设置为 tokenizer.eos_token。

print("🔧 配置 Tokenizer...")

if tokenizer.pad_token_id is None:
    print("⚠️ Tokenizer 没有定义 pad_token_id。设置 pad_token 为 eos_token。")
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ pad_token 已设置为: {tokenizer.pad_token}")
else:
    print(f"✅ Tokenizer 已有 pad_token_id: {tokenizer.pad_token_id}")

print(f"   bos_token: {tokenizer.bos_token}")
print(f"   eos_token: {tokenizer.eos_token}")

# 确保模型在推理模式
model.eval()
print("\n✅ 模型已设置为推理模式 (eval)。")

# 确认模型类型
from peft import PeftModel
if isinstance(model, PeftModel):
    print(f"✅ 确认模型类型: PEFT 模型已正确加载")
else:
    print("✅ 模型已加载，准备进行推理。")


🔧 配置 Tokenizer...
✅ Tokenizer 已有 pad_token_id: 32016
   bos_token: <s>
   eos_token: </s>

✅ 模型已设置为推理模式 (eval)。
✅ 确认模型类型: PEFT 模型已正确加载


In [ ]:
import json as json_module # Changed to json_module to avoid conflict with the json library being imported later
import torch

SYSTEM_MSG = (
    "You are a GIS step instruction parser. "
    "Given a natural language instruction, output a JSON object "
    "describing the corresponding GIS step."
)

def format_prompt(instruction: str, output_dict: dict = None, training: bool = False) -> str:
    """
    训练格式：
    <system>\n<instruction>\n### Response:\n<json>

    推理时令 training=False，不附加 Response 部分。
    """
    output_str = ""
    if output_dict is not None:
        output_str = json_module.dumps(output_dict, ensure_ascii=False)

    if training:
        return (
            f"### System:\n{SYSTEM_MSG}\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_str}"
        )
    else:
        return (
            f"### System:\n{SYSTEM_MSG}\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n"
        )


def infer_sample(instruction, max_tokens=1024): # Increased max_tokens for longer generations
    """GPU推理函数"""

    DEBUG_INFERENCE = False # Temporarily enable for debugging

    # 使用统一的 format_prompt 函数构建输入
    prompt = format_prompt(instruction, training=False)

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH, # Use MAX_LENGTH defined earlier
        padding=True,          # Changed from "max_length" to True for flexible padding
        return_attention_mask=True
    )

    if DEBUG_INFERENCE:
        print(f"DEBUG: Prompt tokens (input_ids): {inputs['input_ids'][0].tolist()}")
        print(f"DEBUG: Prompt text (decoded): {tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=False)}")


    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # 生成
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )

    if DEBUG_INFERENCE:
        print(f"DEBUG: Raw model outputs (tensor): {outputs[0].tolist()}")
        print(f"DEBUG: Input tokens length: {len(inputs['input_ids'][0])}")
        print(f"DEBUG: Generated tokens slice: {outputs[0][len(inputs['input_ids'][0]):].tolist()}")


    # 解码
    # 解码时需要确保只解码新生成的token，而不是整个序列
    # 从原始输入id的长度开始解码
    generated_text = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=False)

    if DEBUG_INFERENCE:
        print(f"DEBUG: Generated text (before strip): '{generated_text}'")

    # --- MODIFIED JSON EXTRACTION LOGIC ---
    # The model often generates additional text, prompts, or separators after the intended JSON.
    # We need to extract only the first, complete JSON object.

    json_str_candidate = generated_text.strip()

    # Find the start of the first potential JSON object
    first_brace_idx = json_str_candidate.find('{')

    if first_brace_idx == -1:
        # No opening brace found, return empty or the raw text as a failure
        json_part = json_str_candidate
    else:
        # Start from the first opening brace
        json_str_candidate = json_str_candidate[first_brace_idx:]

        # Look for typical end markers to cut off extraneous generation
        # Add EOS token to the end markers
        end_markers = ["\n\n### Instruction:", "---", tokenizer.eos_token if tokenizer.eos_token else "<|endoftext|>"]
        effective_end_idx = len(json_str_candidate)

        for marker in end_markers:
            idx = json_str_candidate.find(marker)
            if idx != -1 and idx < effective_end_idx:
                effective_end_idx = idx

        json_str_candidate = json_str_candidate[:effective_end_idx].strip()

        # Try to find the last closing brace for a complete JSON
        # This is an iterative approach to find the correct closing brace,
        # especially if there are nested braces or incomplete JSONs.
        balance = 0
        last_valid_brace_idx = -1
        for i, char in enumerate(json_str_candidate):
            if char == '{':
                balance += 1
            elif char == '}':
                balance -= 1
            if balance == 0 and char == '}': # Found a balanced closing brace
                last_valid_brace_idx = i
                break  # <--- STOP AT THE FIRST COMPLETE JSON OBJECT

        if last_valid_brace_idx != -1:
            json_part = json_str_candidate[:last_valid_brace_idx + 1]
        else:
            # If no balanced closing brace found, try to use the last one found.
            # This is less robust but might catch some cases.
            last_brace_idx = json_str_candidate.rfind('}')
            if last_brace_idx != -1:
                json_part = json_str_candidate[:last_brace_idx + 1]
            else:
                json_part = json_str_candidate # Fallback: return as is

    return json_part


In [ ]:
print("🎯 交互式模型测试\n")
print("使用下面的代码自定义GIS指令来测试模型")
print("="*70)

# 自定义输入 - 修改这些变量来测试不同的指令
your_instruction = "Delete E Probleem Object"  # 修改这里：输入你的GIS指令

print(f"\n📝 你的输入:")
print(f"   指令: {your_instruction}")

print(f"\n⏳ 生成中...\n")

# 生成结果
result = infer_sample(
    instruction=your_instruction,
    max_tokens=1024 # Increased max_tokens for longer generations
)

print("📤 模型输出:")
print("-" * 70)
print(result)
print("-" * 70)

# 尝试解析和美化JSON结果
try:
    import json
    json_result = json.loads(result)

    print("\n✅ JSON解析成功！")
    print("\n📊 结构化数据:")
    print(json.dumps(json_result, indent=2, ensure_ascii=False))

    # 提取关键信息
    print("\n🔍 关键信息提取:")
    for key, value in json_result.items():
        if not isinstance(value, (dict, list)):
            print(f"   • {key}: {value}")
        elif isinstance(value, list):
            print(f"   • {key}: [{len(value)} 项]")

except json.JSONDecodeError as e:
    print(f"\n⚠️  JSON解析失败")
    print(f"   错误: {str(e)[:100]}")
    print(f"\n💡 提示:")
    print(f"   • 模型可能需要更多训练来生成有效的JSON")
    print(f"   • 增加BATCH_SIZE或训练轮数可能会改善结果")

print("\n" + "="*70)
print("💡 提示: 修改上面代码中的 your_instruction 来测试其他指令")


🎯 交互式模型测试

使用下面的代码自定义GIS指令来测试模型

📝 你的输入:
   指令: Delete E Probleem Object

⏳ 生成中...

📤 模型输出:
----------------------------------------------------------------------
{"module": "Datamodel CRUD", "method": "Delete", "object": "E Probleem Object", "database": "elektra", "command": "Execute Datamodel CRUD Testcommand", "object_id": "1231231", "test_data": {"create": {}, "update": {}, "editor": {}}}
----------------------------------------------------------------------

✅ JSON解析成功！

📊 结构化数据:
{
  "module": "Datamodel CRUD",
  "method": "Delete",
  "object": "E Probleem Object",
  "database": "elektra",
  "command": "Execute Datamodel CRUD Testcommand",
  "object_id": "1231231",
  "test_data": {
    "create": {},
    "update": {},
    "editor": {}
  }
}

🔍 关键信息提取:
   • module: Datamodel CRUD
   • method: Delete
   • object: E Probleem Object
   • database: elektra
   • command: Execute Datamodel CRUD Testcommand
   • object_id: 1231231

💡 提示: 修改上面代码中的 your_instruction 来测试其他指令


{
  "testdbs0_4": ":schema_elektra",
  "testobjs0_4": "S DMS Worldmap",
  "testobj_ids0_4": null,
  "testmodules0_4": "Datamodel CRUD",
  "testmethodes0_4": "Delete",
  "testcommands0_4": "Execute Datamodel CRUD Testcommand",
  "testdata_cr0_4": {},
  "testdata_upd0_4": {},
  "testdata_editor0_4": {},
  "testcases": ["Datamodel CRUD Object"]
  }
